In [1]:
# Use the "Python 3.14 (theochem2026)" kernel for this notebook.
import sys
print(sys.executable)

/Users/rolandmitric/WORK/GITHUB/master_programming_2026/.venv/bin/python


In [2]:
from theochem2026 import Atom, Molecule

In [3]:
benzene_xyz = """6
Benzene molecule
C 0.000000 1.402720 0.000000
C 1.214790 0.701360 0.000000
C 1.214790 -0.701360 0.000000
C 0.000000 -1.402720 0.000000
C -1.214790 -0.701360 0.000000
C -1.214790 0.701360 0.000000
H 0.000000 2.490290 0.000000
H 2.156660 1.245150 0.000000
H 2.156660 -1.245150 0.000000
H 0.000000 -2.490290 0.000000
H -2.156660 -1.245150 0.000000
H -2.156660 1.245150 0.000000
"""

benzene = Molecule.from_string(benzene_xyz)


In [4]:
import pyvista as pv
pv.set_jupyter_backend("trame")
print(pv.__version__)

0.47.3


In [5]:
import aiohttp
import trame
print(aiohttp.__version__)

3.13.5


In [6]:
from dataclasses import dataclass, field
import pyvista as pv
COLOR = {
    "C": "green",
    "H": "white",
}
ATOM_RADIUS = {
    "C": 0.4,
    "H": 0.2,
}
COVALENT_RADIUS = {
    "C": 0.77,
    "H": 0.37,
}
@dataclass
class Visualization:
    molecule: Molecule
    plotter: pv.Plotter = pv.Plotter()
    def visualize(self):
        #await launch_server().ready
        for atom in self.molecule.atoms:
            x, y, z = atom.coord
            sphere = pv.Sphere(radius=ATOM_RADIUS.get(atom.symbol, 0.4),
                               center=(x, y, z), 
                               theta_resolution=48,
                               phi_resolution=48)
            self.plotter.add_mesh(sphere, color=COLOR.get(atom.symbol, "white"),smooth_shading=True)

        for i, atom in enumerate(self.molecule.atoms):
            for j in range(i + 1, len(self.molecule.atoms)):
                other_atom = self.molecule.atoms[j]
                distance = atom.get_distance(other_atom)
                cov_rad_sum = COVALENT_RADIUS.get(atom.symbol, 0.77) + COVALENT_RADIUS.get(other_atom.symbol, 0.77)
                if abs(distance - cov_rad_sum) < 0.2:  # Threshold for bonding
                    center = (atom.coord + other_atom.coord) / 2
                    direction = other_atom.coord - atom.coord
                    bond = pv.Cylinder(center=center,
                                       direction=direction,
                                       radius=0.1,
                                       height=distance,
                                       resolution=48)
                    self.plotter.add_mesh(bond, color="gray", smooth_shading=True)
        self.plotter.show()


visualization = Visualization(benzene)
visualization.visualize()

Widget(value='<iframe src="http://localhost:57007/index.html?ui=P_0x14ce7cec0_0&reconnect=auto" class="pyvista…